# WarpFactory Interactive Explorer

Interactive exploration of warp-drive metrics with ipywidgets. This is the
primary interactive UI for WarpFactory (the Qt GUI is maintenance-only).

Requires the `jupyter` extra:

```bash
pip install ".[jupyter]"
```

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/JohnsterID/WarpFactory/main?labpath=examples%2Finteractive_explorer.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnsterID/WarpFactory/blob/main/examples/interactive_explorer.ipynb)

On Google Colab (or any environment without the package installed), run:

```
!pip install "warpfactory[jupyter] @ git+https://github.com/JohnsterID/WarpFactory"
```


## 1. Live explorer

Pick a metric, drag the sliders, and watch the metric component, Eulerian
energy density, and causal structure update. The status line reports the
four pointwise energy conditions and the Ford-Roman quantum inequality
verdict for the current bubble parameters.


In [ ]:
from warpfactory.interactive import JupyterExplorer

explorer = JupyterExplorer()
explorer.display()

## 2. Scripting the same pipeline

The explorer is a thin front end over `ExplorerModel`, which you can drive
directly for reproducible, scripted analysis.


In [ ]:
import numpy as np

from warpfactory.interactive import ExplorerModel

model = ExplorerModel(x=np.linspace(-8.0, 8.0, 400))
result = model.evaluate(
    "Alcubierre", {"v_s": 2.0, "R": 1.0, "sigma": 4.0}, diagnostics=True
)

print("Energy conditions:", result.conditions)
print("Quantum inequality:", result.quantum_inequality)

## 3. Parameter sweeps

Sweep the bubble speed and track the peak (negative) energy density.


In [ ]:
import matplotlib.pyplot as plt

speeds = np.linspace(0.5, 5.0, 10)
peak_density = []
for v_s in speeds:
    sweep = model.evaluate("Alcubierre", {"v_s": v_s, "R": 1.0, "sigma": 4.0})
    peak_density.append(np.abs(sweep.stress_energy["T_tt"]).max())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(speeds, peak_density, "o-")
ax.set_xlabel("bubble speed $v_s$ [c]")
ax.set_ylabel(r"peak $|T_{tt}|$ [1/m$^2$]")
ax.set_title("Wall energy density grows with bubble speed")
plt.show()

## 4. Comparing metrics

Every catalog metric runs through the identical pipeline, so comparisons
are apples-to-apples.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name in ["Alcubierre", "Lentz", "Van Den Broeck"]:
    comparison = model.evaluate(name)
    ax.plot(comparison.x, comparison.stress_energy["T_tt"], label=name)
ax.set_xlabel("x")
ax.set_ylabel(r"$T_{tt}$ [1/m$^2$]")
ax.set_title("Eulerian energy density by metric (catalog defaults)")
ax.legend()
plt.show()